# Análisis Exploratorio de Datos (EDA) – Auto MPG
## Tres personas / tres bloques temáticos

**Dataset:** Auto MPG (disponible en Kaggle como `uciml/autompg-dataset` y originalmente en UCI Machine Learning Repository).

Contiene especificaciones técnicas de 398 automóviles y su consumo de combustible (millas por galón).

| Variable | Descripción |
|----------|-------------|
| mpg | Millas por galón (variable objetivo) |
| cylinders | Número de cilindros |
| displacement | Cilindrada del motor (pulgadas cúbicas) |
| horsepower | Potencia del motor |
| weight | Peso del vehículo (libras) |
| acceleration | Tiempo de 0 a 60 mph (segundos) |
| model_year | Año del modelo (70-82 → 1970-1982) |
| origin | Origen: 1 = USA, 2 = Europa, 3 = Japón |
| car_name | Nombre del vehículo |

---
### Estructura del notebook
1. **Persona 1** – EDA inicial vs avanzado y relaciones numéricas  
2. **Persona 2** – Correlación, matrices, heatmaps y variables categóricas  
3. **Persona 3** – Atípicos, grupos, visualizaciones avanzadas y 3 hallazgos finales

## 0. Carga de librerías y del dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

# Configuración visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Descarga automática desde Kaggle
path = kagglehub.dataset_download("uciml/autompg-dataset")
df = pd.read_csv(os.path.join(path, "auto-mpg.csv"))

# Normalizar nombres de columnas (el CSV de Kaggle usa espacios)
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

# Mapeo de origen para mayor legibilidad
df["origin_label"] = df["origin"].map({1: "USA", 2: "Europa", 3: "Japón"})

print("Columnas:", list(df.columns))
print("Forma del dataset:", df.shape)
df.head()

---
# PERSONA 1 – EDA y relaciones numéricas

En este bloque se realiza el EDA inicial, se exploran las relaciones entre variables numéricas con diagramas de dispersión (Matplotlib y Seaborn) y se analiza la relación entre peso, potencia, cilindrada y MPG.

### 1.1 Diferencia entre EDA inicial y EDA avanzado

**EDA inicial (o descriptivo básico):**
- Forma del dataset (filas × columnas)
- Tipos de datos (`dtypes`)
- Valores faltantes
- Estadísticas descriptivas (`describe()`)
- Distribuciones univariadas (histograma, boxplot simple)

**EDA avanzado:**
- Relaciones bivariadas y multivariadas
- Correlaciones y dependencias
- Segmentación por grupos (categóricas)
- Detección de patrones, outliers y posibles interacciones
- Formulación de hipótesis para modelado posterior

In [ ]:
# --- EDA INICIAL ---
print("=== Información general ===")
df.info()
print("\n=== Valores nulos ===")
print(df.isnull().sum())
print("\n=== Estadísticas descriptivas (numéricas) ===")
df.describe().round(2)

In [ ]:
# Tratamiento simple de nulos en horsepower (media)
# Primero convertimos la columna a tipo numérico, transformando los valores no numéricos (como '?') en NaN
df["horsepower"] = pd.to_numeric(df["horsepower"], errors="coerce")

# Ahora que es numérica, rellenamos los valores nulos con la media
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].mean())
print("Nulos restantes en horsepower:", df["horsepower"].isnull().sum())

### 1.2 Relaciones entre variables numéricas – Diagramas de dispersión (Matplotlib)

In [ ]:
# Scatter plot clásico con Matplotlib: peso vs MPG
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df["weight"], df["mpg"], alpha=0.6, edgecolors="k", linewidth=0.4)
ax.set_xlabel("Peso (libras)")
ax.set_ylabel("MPG (millas por galón)")
ax.set_title("Relación entre Peso y Consumo (MPG) – Matplotlib")
plt.tight_layout()
plt.show()

In [ ]:
# Varios scatter plots en subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].scatter(df["weight"], df["mpg"], alpha=0.5, c="steelblue")
axes[0].set_xlabel("Peso")
axes[0].set_ylabel("MPG")
axes[0].set_title("Peso vs MPG")

axes[1].scatter(df["horsepower"], df["mpg"], alpha=0.5, c="darkorange")
axes[1].set_xlabel("Potencia (HP)")
axes[1].set_ylabel("MPG")
axes[1].set_title("Potencia vs MPG")

axes[2].scatter(df["displacement"], df["mpg"], alpha=0.5, c="seagreen")
axes[2].set_xlabel("Cilindrada")
axes[2].set_ylabel("MPG")
axes[2].set_title("Cilindrada vs MPG")

plt.suptitle("Relaciones numéricas clave con Matplotlib", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 1.3 Scatterplot con Seaborn e incorporación de variables categóricas

In [ ]:
# Scatter con Seaborn coloreado por origen
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="weight",
    y="mpg",
    hue="origin_label",
    style="origin_label",
    s=70,
    alpha=0.75
)
plt.title("Peso vs MPG coloreado por Origen del vehículo")
plt.xlabel("Peso (libras)")
plt.ylabel("MPG")
plt.legend(title="Origen")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter con tamaño según potencia y color por cilindros
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="weight",
    y="mpg",
    size="horsepower",
    hue="cylinders",
    sizes=(40, 250),
    alpha=0.7,
    palette="viridis"
)
plt.title("Peso vs MPG – tamaño = potencia, color = cilindros")
plt.xlabel("Peso (libras)")
plt.ylabel("MPG")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 1.4 Aplicación: relación entre peso, potencia, cilindrada y MPG

In [ ]:
# Matriz de dispersión (pairplot) de las 4 variables clave
vars_clave = ["mpg", "weight", "horsepower", "displacement"]
sns.pairplot(df[vars_clave], diag_kind="kde", corner=True, plot_kws={"alpha": 0.6})
plt.suptitle("Relaciones entre peso, potencia, cilindrada y MPG", y=1.02)
plt.show()

In [ ]:
# Resumen de correlaciones de estas variables
print("Correlaciones de Pearson entre las variables clave:")
df[vars_clave].corr().round(3)

**Observación:**  
Existe una fuerte relación **inversa** entre peso, potencia y cilindrada con el MPG. Cuanto más pesado, más potente o de mayor cilindrada es el coche, peor es su eficiencia de combustible.

---
# PERSONA 2 – Correlación y variables categóricas

En este bloque se trabaja la correlación de Pearson, la matriz de correlaciones, el mapa de calor, la identificación de correlaciones fuertes y las tablas cruzadas entre variables categóricas.

### 2.1 Concepto e interpretación de correlación

- La **correlación de Pearson** mide la fuerza y dirección de la relación **lineal** entre dos variables numéricas.
- Rango: de **-1** (relación inversa perfecta) a **+1** (relación directa perfecta). 0 = sin relación lineal.
- Interpretación orientativa:
  - |r| < 0.3 → débil
  - 0.3 ≤ |r| < 0.7 → moderada
  - |r| ≥ 0.7 → fuerte

In [ ]:
# Matriz de correlaciones (solo variables numéricas)
num_cols = ["mpg", "cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]
corr_matrix = df[num_cols].corr()
corr_matrix.round(3)

### 2.2 Mapa de calor (Heatmap)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
plt.title("Mapa de calor – Matriz de correlaciones (Pearson)")
plt.tight_layout()
plt.show()

### 2.3 Identificación automática de correlaciones fuertes

In [ ]:
# Extraer pares con |correlación| >= 0.7 (excluyendo la diagonal)
threshold = 0.7
strong_pairs = (
    corr_matrix
    .where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
strong_pairs.columns = ["Variable_1", "Variable_2", "Correlacion"]
strong_pairs = strong_pairs[strong_pairs["Correlacion"].abs() >= threshold]
strong_pairs = strong_pairs.sort_values("Correlacion", key=abs, ascending=False)

print(f"Pares con |correlación| ≥ {threshold}:")
strong_pairs

### 2.4 Relaciones entre variables categóricas – Tablas cruzadas

In [ ]:
# Crear una variable categórica de eficiencia
df["eficiencia"] = pd.cut(
    df["mpg"],
    bins=[0, 18, 25, 50],
    labels=["Baja", "Media", "Alta"]
)

# Tabla cruzada: origen × eficiencia (conteos)
ct_counts = pd.crosstab(df["origin_label"], df["eficiencia"])
print("=== Frecuencias absolutas ===")
ct_counts

In [ ]:
# Porcentajes por fila (distribución de eficiencia dentro de cada origen)
ct_pct = pd.crosstab(df["origin_label"], df["eficiencia"], normalize="index") * 100
print("=== Porcentajes por origen (%) ===")
ct_pct.round(1)

In [ ]:
# Otra tabla cruzada interesante: cilindros × origen
ct_cyl = pd.crosstab(df["cylinders"], df["origin_label"], margins=True)
print("=== Cilindros vs Origen (con totales) ===")
ct_cyl

**Observación:**  
Los vehículos japoneses y europeos concentran la mayor proporción de eficiencia alta, mientras que los estadounidenses dominan en la categoría de baja eficiencia. Además, los motores de 8 cilindros son casi exclusivos de origen USA.

---
# PERSONA 3 – Atípicos, grupos y conclusiones

En este bloque se visualizan las tablas cruzadas, se detectan valores atípicos con la regla del IQR, se analizan patrones por grupos y se documentan los tres hallazgos finales del análisis.

### 3.1 Visualización de tablas cruzadas

In [ ]:
# Heatmap de la tabla cruzada origen × eficiencia
plt.figure(figsize=(8, 5))
sns.heatmap(ct_counts, annot=True, fmt="d", cmap="YlGnBu", linewidths=0.5)
plt.title("Frecuencias: Origen × Nivel de eficiencia")
plt.ylabel("Origen")
plt.xlabel("Eficiencia")
plt.tight_layout()
plt.show()

In [ ]:
# Barplot apilado de porcentajes
ct_pct.plot(kind="bar", stacked=True, colormap="Set2", figsize=(9, 5))
plt.title("Distribución porcentual de eficiencia por origen")
plt.ylabel("Porcentaje (%)")
plt.xlabel("Origen")
plt.xticks(rotation=0)
plt.legend(title="Eficiencia", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

### 3.2 Concepto de valores atípicos y regla del IQR

Un **valor atípico (outlier)** es una observación que se desvía notablemente del resto de los datos.

**Regla del IQR (rango intercuartílico):**
- Q1 = percentil 25, Q3 = percentil 75
- IQR = Q3 − Q1
- Límite inferior = Q1 − 1.5 × IQR
- Límite superior = Q3 + 1.5 × IQR
- Cualquier valor fuera de estos límites se considera atípico.

In [ ]:
def detectar_outliers_iqr(serie):
    """Devuelve los índices de los outliers según la regla 1.5*IQR."""
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    outliers = serie[(serie < lim_inf) | (serie > lim_sup)]
    return outliers, lim_inf, lim_sup

# Aplicar a MPG y a horsepower
out_mpg, li_mpg, ls_mpg = detectar_outliers_iqr(df["mpg"])
out_hp, li_hp, ls_hp = detectar_outliers_iqr(df["horsepower"])

print(f"Outliers en MPG (límites {li_mpg:.1f} – {ls_mpg:.1f}): {len(out_mpg)}")
print(out_mpg.values)
print(f"\nOutliers en Horsepower (límites {li_hp:.1f} – {ls_hp:.1f}): {len(out_hp)}")
print(out_hp.values[:10], "..." if len(out_hp) > 10 else "")

### 3.3 Boxplot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(y=df["mpg"], ax=axes[0], color="skyblue")
axes[0].set_title("Boxplot de MPG")
axes[0].set_ylabel("MPG")

sns.boxplot(x="origin_label", y="mpg", data=df, ax=axes[1], palette="Set2")
axes[1].set_title("Boxplot de MPG por Origen")
axes[1].set_xlabel("Origen")
axes[1].set_ylabel("MPG")

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot de potencia por número de cilindros
plt.figure(figsize=(9, 5))
sns.boxplot(x="cylinders", y="horsepower", data=df, palette="viridis")
plt.title("Distribución de potencia según número de cilindros")
plt.xlabel("Cilindros")
plt.ylabel("Horsepower")
plt.tight_layout()
plt.show()

### 3.4 Patrones relevantes por grupos – Barplot y Pairplot

In [ ]:
# Barplot: MPG medio por origen y por año
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(x="origin_label", y="mpg", data=df, ax=axes[0], palette="Set2", errorbar="sd")
axes[0].set_title("MPG promedio ± DE por Origen")
axes[0].set_xlabel("Origen")
axes[0].set_ylabel("MPG medio")

sns.barplot(x="model_year", y="mpg", data=df, ax=axes[1], color="steelblue", errorbar=None)
axes[1].set_title("MPG promedio por Año del modelo")
axes[1].set_xlabel("Año (19xx)")
axes[1].set_ylabel("MPG medio")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Pairplot segmentado por origen (muestra de variables)
sns.pairplot(
    df,
    vars=["mpg", "weight", "horsepower"],
    hue="origin_label",
    diag_kind="kde",
    corner=True,
    plot_kws={"alpha": 0.65, "s": 40}
)
plt.suptitle("Pairplot segmentado por Origen", y=1.02)
plt.show()

### 3.5 Formulación y documentación de hallazgos exploratorios

In [ ]:
# Algunos números clave para respaldar los hallazgos
print("=== Resumen numérico de apoyo ===")
print("\nMPG medio por origen:")
print(df.groupby("origin_label")["mpg"].agg(["mean", "median", "std"]).round(2))

print("\nCorrelación peso-MPG:", round(df["weight"].corr(df["mpg"]), 3))
print("Correlación horsepower-MPG:", round(df["horsepower"].corr(df["mpg"]), 3))
print("Correlación displacement-MPG:", round(df["displacement"].corr(df["mpg"]), 3))

print("\nEvolución del MPG medio (primeros vs últimos años):")
print("1970-72:", round(df[df["model_year"].isin([70, 71, 72])]["mpg"].mean(), 2))
print("1980-82:", round(df[df["model_year"].isin([80, 81, 82])]["mpg"].mean(), 2))

### 3.6 Los 3 hallazgos finales del análisis de Auto MPG

---

**Hallazgo 1 – Relación inversa fuerte entre peso/potencia/cilindrada y eficiencia**  
El peso del vehículo, la potencia del motor y la cilindrada muestran correlaciones negativas fuertes con el MPG (alrededor de −0.83, −0.78 y −0.80 respectivamente).  
Los coches más pesados y potentes consumen significativamente más combustible. Esta es la relación más clara y consistente del dataset.

---

**Hallazgo 2 – El origen geográfico marca diferencias importantes de eficiencia**  
Los vehículos de origen japonés presentan el MPG medio más alto, seguidos por los europeos. Los estadounidenses, en cambio, concentran la mayoría de los modelos de baja eficiencia y de motores de 8 cilindros.  
La tabla cruzada origen × eficiencia confirma que Japón y Europa dominan la categoría “Alta”, mientras que USA domina la “Baja”.

---

**Hallazgo 3 – Mejora temporal de la eficiencia (1970-1982)**  
Se observa una tendencia clara de aumento del MPG a lo largo de los años del modelo. Los vehículos de principios de los 70 tienen un consumo medio notablemente peor que los de principios de los 80.  
Esto refleja, entre otras cosas, la respuesta de la industria a las crisis del petróleo y a las regulaciones de eficiencia de la época.

---

Estos tres hallazgos resumen el comportamiento principal del dataset y sirven como base sólida para cualquier modelado predictivo posterior (regresión de MPG).